# Lab 2 - Notebook 3: Rerank (The Lift)

**Goal:** apply Cohere Rerank-v4 on top of vector search and measure the improvement.

**Concept:** vector search is fast but coarse. Rerank is a *cross-encoder*: it reads
the query and each candidate document *together* and scores true relevance. We use
vector search to get a candidate shortlist, then Rerank to order it well.

Watch for two things, not just one:
1. The correct answer **climbs** in the ranking.
2. The scores **sharpen**: vector search scores cluster in a vague 0.3 to 0.4 band;
   rerank scores jump to 0.9+ with clear separation. Rerank turns 'these all seem
   vaguely relevant' into 'this one, definitively.'


## 1. Load everything

As before, this notebook runs in its own fresh session, so we re-import the helpers
and re-load the corpus and the saved vectors. Nothing carries over from the previous
notebooks automatically.


In [ ]:
from cohere_helpers import embed_texts, cosine_search, rerank
import json, numpy as np

with open('support_corpus.json', 'r', encoding='utf-8') as f:
    corpus = json.load(f)
doc_vectors = np.load('doc_vectors.npy')
doc_texts = [f"{d['title']}. {d['content']}" for d in corpus]
print('Ready.')


## 2. The before-and-after function

For a query we will:
1. Get the top candidates by vector search (the 'before').
2. Pass those same candidates through Rerank (the 'after').
3. Print them side by side so you can compare rank and score.


In [ ]:
def before_after(query, shortlist=10, show_k=5):
    # BEFORE: vector search
    qv = embed_texts([query], input_type='search_query')[0]
    vec = cosine_search(qv, doc_vectors, top_k=shortlist)
    cand_idx = [i for i, _ in vec]
    cand_texts = [doc_texts[i] for i in cand_idx]

    # AFTER: rerank the same candidates
    rr = rerank(query, cand_texts, top_n=show_k)

    print(f'QUERY: {query}\n')
    print('BEFORE (vector search):')
    for rank, (i, s) in enumerate(vec[:show_k], 1):
        print(f'  {rank}. [{s:.3f}] {corpus[i]["title"]}')
    print('\nAFTER (rerank):')
    for rank, r in enumerate(rr, 1):
        orig = cand_idx[r['index']]
        print(f"  {rank}. [{r['relevance_score']:.3f}] {corpus[orig]['title']}")
    print('-' * 70)


## 3. The lift: run all five demo queries

Compare BEFORE (vector) and AFTER (rerank) for each query. Notice:
- Some correct answers climb to position 1.
- One query (deployment) is already correct in vector search; rerank keeps it correct
  and does no harm. Not every query needs rerank, and that is an honest, useful result.
- The rate-limit query is the strongest case: vector search misses the right answer
  entirely, and rerank rescues it.

If the output is truncated, click "scrollable element" to view all of it.


In [ ]:
demo_queries = [
    'I cannot log in after changing my password',
    'Why is my bill higher than expected this month',
    'My deployment keeps failing',
    'How do I get my data out of the platform',
    'I keep getting rate limited with 429 errors',
]

for q in demo_queries:
    before_after(q)


## 4. Your turn

Try your own query. In the code cell below, **replace the text inside the quotes**
with your own question. Phrase it the way a frustrated customer would, using different
words than the documentation uses. Then run the cell and watch how rerank handles the
mismatch between your wording and the document's wording.

For example, change:

`before_after('the website wont take my new login details')`

to something like:

`before_after('I was charged twice this month')`

If you added a query in Notebook 2, try that same one here and compare.


In [ ]:
# Replace the text inside the quotes with your own question, then run this cell.
before_after('the website wont take my new login details')


## What you learned

- **Embed-v4** turns text into vectors for fast semantic search.
- **Vector search** is a strong first pass but coarse: it can rank near-misses high,
  and sometimes misses the right answer entirely.
- **Rerank-v4** reads query and document together and fixes the ordering, and it does
  so with high confidence (scores jump from the ambiguous 0.3-0.4 band to 0.9+).
- Rerank **rescues the hard cases without breaking the easy ones**: where vector search
  already won, rerank keeps it correct.
- The pattern **embed -> vector search -> rerank** is the production recipe for
  high-quality retrieval, and the foundation of good RAG.

### Scaling beyond the lab
We used NumPy for vector search because the corpus is small (~150 docs). At
production scale (millions of documents) you would swap the NumPy step for a
vector database like **FAISS** or **Azure AI Search**. The embed and rerank calls
stay exactly the same.
